### Импорты

In [67]:
import talib
from pybit.unified_trading import HTTP
import datetime as dt
import time
import backtesting
import backtrader as bt
import pandas as pd
import mplfinance as mpf
session = HTTP(testnet = False, timeout = 60 )

from tqdm import tqdm

### Загрузка данных

In [76]:
def get_hist_klines(
    session,
    symbol: str,
    interval: int,
    dt_start: dt.datetime,
    dt_end: dt.datetime | None = None,
    limit: int = 1000
) -> pd.DataFrame:

    dt_start_unix = int(dt_start.timestamp() * 1000)

    if dt_end is None:
        dt_end_unix = int(dt.datetime.now().timestamp() * 1000)
    else:
        dt_end_unix = int(dt_end.timestamp() * 1000)

    candles_total_lst = []

    while dt_end_unix > dt_start_unix:

        data = session.get_kline(
            category="linear",
            symbol=symbol,
            interval=interval,
            limit=limit,
            end=dt_end_unix
        )

        candles_current_lst = data['result']['list']

        if not candles_current_lst:
            break

        candles_total_lst.extend(
            candle
            for candle in candles_current_lst
            if int(candle[0]) >= dt_start_unix
        )

        # timestamp самой старой свечи
        dt_end_unix = int(candles_current_lst[-1][0]) - 1

    df = pd.DataFrame(candles_total_lst, columns=['date', 'open', 'high', 'low', 'close', 'volume', 'turnover' ] )
    df['date'] = pd.to_datetime(
    df['date'].astype('int64'),
    unit='ms')

    # строки -> числа
    cols = ['open', 'high', 'low', 'close', 'volume', 'turnover']

    df[cols] = df[cols].apply(pd.to_numeric)

    # date становится индексом
    df.set_index('date', inplace=True)

    # сортировка по времени
    df.sort_index(inplace=True)

    return df

In [77]:
df = get_hist_klines(
    session = session,
    symbol = 'BTCUSDT',
    interval = 5,
    dt_start = dt.datetime(2026, 2, 1, 20, 0) )

df

,open,high,low,close,volume,turnover
date,,,,,,
2026-02-01 17:00:00,77456.1,77596.7,77367.9,77596.7,180.033,1.394773e+07
2026-02-01 17:05:00,77596.7,77715.0,77544.9,77610.7,403.363,3.131887e+07
2026-02-01 17:10:00,77610.7,77634.7,77505.1,77598.8,76.237,5.914285e+06
2026-02-01 17:15:00,77598.8,77715.0,77523.9,77676.9,226.026,1.754917e+07
2026-02-01 17:20:00,77676.9,77687.7,77300.0,77380.9,211.730,1.639169e+07
...,...,...,...,...,...,...
2026-06-14 17:00:00,63913.1,64000.0,63892.6,63984.4,171.196,1.094541e+07
2026-06-14 17:05:00,63984.4,63984.5,63890.5,63908.8,114.142,7.295890e+06
2026-06-14 17:10:00,63908.8,63949.5,63906.6,63933.7,75.446,4.823357e+06


In [71]:
print(df.index)

print(type(df.index[0]))

RangeIndex(start=0, stop=38308, step=1)
<class 'int'>


In [63]:
class BreakoutStrategy(bt.Strategy):

    # Параметры стратегии
    params = dict(

        # Количество свечей, максимум которых будем пробивать
        lookback=20,

        # Риск на одну сделку (1% от депозита)
        risk_pct=0.01,

        # Размер стопа (1%)
        stop_pct=0.01,

        # Risk Reward
        # тейк = стоп × rr
        rr=2
    )

    def __init__(self):

        # Highest будет хранить максимум
        # за последние lookback свечей
        self.highest = bt.ind.Highest(
            self.data.high,
            period=self.p.lookback
        )

        # Переменная для хранения текущего ордера
        # нужна, чтобы не отправить несколько ордеров подряд
        self.order = None

        # Цены стопа и тейка
        self.stop_price = None
        self.take_price = None

    def next(self):
        """
        Вызывается на каждой новой свече.
        Здесь находится вся логика стратегии.
        """

        # Если ордер уже отправлен и ещё не исполнен,
        # ничего не делаем
        if self.order:
            return

        # --------------------------------
        # Если позиции сейчас нет
        # --------------------------------
        if not self.position:

            # Проверяем пробой максимума
            # highest[-1] - максимум предыдущих N свечей
            breakout = (
                self.data.close[0]
                > self.highest[-1]
            )

            if breakout:

                # Текущая цена входа
                entry = self.data.close[0]

                # Стоп на расстоянии stop_pct
                self.stop_price = (
                    entry *
                    (1 - self.p.stop_pct)
                )

                # Тейк = стоп × rr
                self.take_price = (
                    entry *
                    (
                        1 +
                        self.p.stop_pct *
                        self.p.rr
                    )
                )

                # Сколько готовы потерять
                # при срабатывании стопа
                risk_amount = (
                    self.broker.getvalue()
                    * self.p.risk_pct
                )

                # Расстояние от входа до стопа
                stop_distance = (
                    entry -
                    self.stop_price
                )

                # Размер позиции
                #
                # Пример:
                #
                # Баланс = 10000$
                # риск = 1%
                # риск_amount = 100$
                #
                # стоп = 1000$
                #
                # size = 100 / 1000 = 0.1 BTC
                size = (
                    risk_amount
                    / stop_distance
                )

                print(
                    f"BUY "
                    f"price={entry:.2f} "
                    f"size={size:.5f}"
                )

                # Отправляем рыночный ордер
                self.order = self.buy(
                    size=size
                )

        # --------------------------------
        # Если позиция уже открыта
        # --------------------------------
        else:

            current_price = self.data.close[0]

            # Сработал стоп
            if current_price <= self.stop_price:

                print("STOP")

                self.order = self.close()

            # Сработал тейк
            elif current_price >= self.take_price:

                print("TAKE")

                self.order = self.close()

    def notify_order(self, order):
        """
        Вызывается при изменении статуса ордера.
        """

        # Ордер полностью исполнен
        if order.status == order.Completed:

            print(
                "ORDER FILLED",
                order.executed.price
            )

        # Ордер завершил жизненный цикл
        if order.status in [
            order.Completed,
            order.Canceled,
            order.Margin,
            order.Rejected
        ]:

            # Освобождаем переменную,
            # чтобы можно было создавать новый ордер
            self.order = None

    def notify_trade(self, trade):
        """
        Вызывается после закрытия сделки.
        """

        if trade.isclosed:

            print(
                f"TRADE CLOSED "
                f"PnL={trade.pnl:.2f} "
                f"PnL after commission={trade.pnlcomm:.2f}"
            )

In [78]:
cerebro = bt.Cerebro()

data = bt.feeds.PandasData(
    dataname=df
)

cerebro.adddata(data)

cerebro.addstrategy(
    BreakoutStrategy
)

# стартовый депозит
cerebro.broker.setcash(10000)

# комиссия Bybit
cerebro.broker.setcommission(
    commission=0.0006
)

results = cerebro.run()

print(
    "Final value:",
    cerebro.broker.getvalue()
)

BUY price=77791.00 size=0.12855
BUY price=77927.00 size=0.12833
BUY price=76468.70 size=0.13077
BUY price=77146.90 size=0.12962
BUY price=77320.10 size=0.12933
BUY price=77745.70 size=0.12862
BUY price=77826.60 size=0.12849
BUY price=77911.50 size=0.12835
BUY price=78091.30 size=0.12806
BUY price=78134.50 size=0.12798
BUY price=78524.40 size=0.12735
BUY price=78985.60 size=0.12661
BUY price=78595.80 size=0.12723
BUY price=78705.00 size=0.12706
BUY price=78878.10 size=0.12678
BUY price=79038.20 size=0.12652
BUY price=78952.40 size=0.12666
BUY price=78756.50 size=0.12697
BUY price=78939.60 size=0.12668
BUY price=78367.00 size=0.12760
BUY price=75480.00 size=0.13249
BUY price=75620.40 size=0.13224
BUY price=76211.20 size=0.13121
BUY price=76887.40 size=0.13006
BUY price=75981.30 size=0.13161
BUY price=76074.50 size=0.13145
BUY price=76224.40 size=0.13119
BUY price=76377.60 size=0.13093
BUY price=76490.50 size=0.13074
BUY price=76677.40 size=0.13042
BUY price=76817.70 size=0.13018
BUY pric

In [80]:
strat = results[0]

print(
    strat.analyzers.trades.get_analysis()
)

print(
    strat.analyzers.drawdown.get_analysis()
)

print(
    strat.analyzers.sharpe.get_analysis()
)

AttributeError: 'ItemCollection' object has no attribute 'trades'